In [1]:
"""
MTEB 数据聚类实验：验证向量偏好权重与数据语料类型的相关性
作者: [你的名字]
日期: 2025-10-21
"""

import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# -----------------------------
# 1. 配置部分
# -----------------------------

# MTEB 任务映射表（简写）
task_abbreviations = {
    'AmazonCounterfactualClassification': 'AmazonCFC',
    'AmazonReviewsClassification': 'AmazonRev',
    'Banking77Classification': 'Banking77',
    'EmotionClassification': 'Emotion',
    'ImdbClassification': 'IMDB',
    'MTOPDomainClassification': 'MTOPDomain',
    'MTOPIntentClassification': 'MTOPIntent',
    'MassiveIntentClassification': 'MassiveInt',
    'MassiveScenarioClassification': 'MassiveSce',
    'ToxicConversationsClassification': 'ToxicConv',
    'TweetSentimentExtractionClassification': 'TweetSen',
    'BiorxivClusteringS2S': 'BiorxivClu',
    'MedrxivClusteringS2S': 'MedrxivClu',
    'TwentyNewsgroupsClustering': '20NewsClu',
    'SprintDuplicateQuestions': 'SprintDupQ',
    'TwitterSemEval2015': 'TwitterSem',
    'TwitterURLCorpus': 'TwitterURL',
    'AskUbuntuDupQuestions': 'AskUbuntu',
    'SciDocsRR': 'SciDocsRR',
    'StackOverflowDupQuestions': 'StackOver',
    'BIOSSES': 'BIOSSES',
    'SICK-R': 'SICK-R',
    'STS12': 'STS12',
    'STS13': 'STS13',
    'STS14': 'STS14',
    'STS15': 'STS15',
    'STS16': 'STS16',
    'STS17': 'STS17',
    'STSBenchmark': 'STSBench',
    'ArguAna': 'ArguAna',
    'CQADupstackEnglishRetrieval': 'CQADupRet',
    'NFCorpus': 'NFCorpus',
    'SCIDOCS': 'SCIDOCS',
    'SciFact': 'SciFact'
}

# 要加载的任务
task_list = list(task_abbreviations.keys())

# 每个任务采样数量
SAMPLE_SIZE = 5000

# embedding 模型
MODEL_NAME = "/home/linkco/exa/models/gte-base"

# -----------------------------
# 2. 加载与采样数据
# -----------------------------
def sample_texts_from_task(task_name, sample_size=5000):
    """
    从 MTEB 任务加载数据并采样文本
    """
    try:
        dataset = load_dataset(f"mteb/{task_name}", split="test", trust_remote_code=True)
    except Exception:
        try:
            dataset = load_dataset(f"mteb/{task_name}", split="train", trust_remote_code=True)
        except Exception:
            print(f"❌ 无法加载任务 {task_name}")
            return []

    # 找出可能的文本字段
    text_keys = [k for k in dataset.column_names if any(x in k.lower() for x in ['text', 'sentence', 'query', 'document'])]
    if not text_keys:
        print(f"⚠️ 任务 {task_name} 没有标准文本字段，跳过。")
        return []

    # 尽量多地取出文本字段
    all_texts = []
    for k in text_keys:
        texts = [t for t in dataset[k] if isinstance(t, str)]
        all_texts.extend(texts)

    # 去重 + 采样
    all_texts = list(set(all_texts))
    if len(all_texts) > sample_size:
        all_texts = random.sample(all_texts, sample_size)
    return all_texts


# -----------------------------
# 3. 计算任务级向量中心
# -----------------------------
print("🔹 加载模型中...")
model = SentenceTransformer(MODEL_NAME)

task_embeddings = []
valid_task_names = []

for task_name in tqdm(task_list, desc="计算任务向量"):
    texts = sample_texts_from_task(task_name, SAMPLE_SIZE)
    if len(texts) < 50:
        continue
    emb = model.encode(texts, batch_size=64, show_progress_bar=False)
    centroid = np.mean(emb, axis=0)
    task_embeddings.append(centroid)
    valid_task_names.append(task_abbreviations[task_name])

task_embeddings = np.vstack(task_embeddings)

# -----------------------------
# 4. 聚类 + 可视化
# -----------------------------
print("\n🔹 执行 KMeans 聚类...")
NUM_CLUSTERS = 6
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42).fit(task_embeddings)
labels = kmeans.labels_

# PCA 降维
pca = PCA(n_components=2)
coords = pca.fit_transform(task_embeddings)

plt.figure(figsize=(10, 8))
plt.scatter(coords[:, 0], coords[:, 1], c=labels, cmap="tab10", s=80, alpha=0.8)
for i, name in enumerate(valid_task_names):
    plt.text(coords[i, 0], coords[i, 1], name, fontsize=8)
plt.title("MTEB 任务级语料语义分布聚类（PCA 可视化）")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.grid(True)
plt.show()

# -----------------------------
# 5. 与向量偏好权重聚类结果对比（若有）
# -----------------------------
# 如果你已有 “向量偏好权重聚类结果标签”，请在此加载为 weight_labels

# 示例假设（随机生成模拟）
# 实际中应替换为你已有的聚类结果，如从实验文件中读取
weight_labels = np.random.randint(0, NUM_CLUSTERS, size=len(valid_task_names))

ari = adjusted_rand_score(weight_labels, labels)
nmi = normalized_mutual_info_score(weight_labels, labels)

print(f"\n📊 一致性指标：")
print(f" - Adjusted Rand Index (ARI): {ari:.3f}")
print(f" - Normalized Mutual Information (NMI): {nmi:.3f}")

print("\n✅ 实验完成！")


🔹 加载模型中...


/home/linkco/anaconda3/envs/longemb/lib/python3.12/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.Op.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()
/home/linkco/anaconda3/envs/longemb/lib/python3.12/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.OnnxFunction.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()
计算任务向量:   3%|▎         | 1/34 [00:00<00:32,  1.03it/s]

❌ 无法加载任务 AmazonCounterfactualClassification
